In [1]:
#importations
import os
import gradio as gr
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [2]:
#initializing GROQ for fast inference and gemini alternative
GROQ_KEY = os.getenv('GROQ_API_KEY')
GEMINI_KEY = os.getenv('GEMINI_API_KEY')

groq = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=GROQ_KEY)
gemini = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/", api_key=GEMINI_KEY)

models = ["gemini-3-flash-preview", "openai/gpt-oss-120b"]
client_models = {"gemini-3-flash-preview": gemini, "openai/gpt-oss-120b": groq}

In [3]:
#system prompt
system_prompt = """
Your task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
Your response will be written to a file called main.cpp and then compiled and executed;
Respond only with C++ code.
Python code to port:

```python
{python}
```
"""

In [4]:
def write_to_file(python):
    with open("main.cpp", "w") as f:
        f.write(python)

In [ ]:
def chat(python, model):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)},
    ]
    try:
        stream = client_models[model].chat.completions.create(model=model, messages=messages, stream=True, max_tokens=)
        response = ""
        for chunk in stream:
            response += chunk.choices[0].delta.content
            yield response.replace("```cpp", "").replace("```", "")
        
        write_to_file(response)
    except Exception as e:
        return f"""Encountered an error during execution.Please try again later.
        Error: {e}
        """

In [6]:
#example code
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [7]:
#gradio UI

with gr.Blocks() as demo:
    with gr.Row():
        python = gr.Textbox(label="Python Code", lines=28, value=pi)
        cpp = gr.Textbox(label="C++ Code", lines=28)
    with gr.Row():
        model_selection = gr.Dropdown(models, label="Select Model", value=models[0])
        convert = gr.Button("Convert")
    convert.click(chat, inputs=[python, model_selection], outputs=[cpp])
demo.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
